# 📊 Data Exploration — SensorAPI

Explore the sensor inventory, data quality, reading distributions, and time coverage
across all sensors in the Aiven PostgreSQL database.

**Sensors**: 14 real sensors (heat pump, PV, household, temperatures)
**Readings**: ~1M time-series records (Jul 2025 – present)
**Location**: Bonn, Germany (50.74°N, 7.10°E)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import func, text

from app.database.database import get_db_session
from app.database.models import Sensor, SensorType, Location, SensorReading

pd.set_option('display.max_columns', None)
print("✅ Imports ready")

✅ Imports ready


## 1. Sensor Inventory

Overview of all sensors, their types, units, locations, and reading counts.

In [2]:
with get_db_session() as db:
    sensors = db.query(Sensor).all()
    rows = []
    for s in sensors:
        count = db.query(func.count(SensorReading.id)).filter(SensorReading.sensor_id == s.id).scalar()
        time_range = db.query(
            func.min(SensorReading.timestamp),
            func.max(SensorReading.timestamp)
        ).filter(SensorReading.sensor_id == s.id).one()
        rows.append({
            'sensor_name': s.name,
            'type': s.sensor_type.name,
            'unit': s.sensor_type.unit,
            'location': s.location.name if s.location else 'N/A',
            'readings': count,
            'first_reading': time_range[0],
            'last_reading': time_range[1],
            'is_active': s.is_active,
        })

df_sensors = pd.DataFrame(rows).sort_values('readings', ascending=False)
df_sensors

2026-03-24 15:17:35,212 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-24 15:17:35,212 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:17:35,238 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-24 15:17:35,238 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:17:35,261 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-24 15:17:35,262 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:17:35,286 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:17:35,288 INFO sqlalchemy.engine.Engine SELECT api_sensors.id AS api_sensors_id, api_sensors.device_id AS api_sensors_device_id, api_sensors.name AS api_sensors_name, api_sensors.description AS api_sensors_description, api_sensors.sensor_type_id AS api_sensors_sensor_type_id, api_sensors.location_id AS api_sensors_location_id, api_sensors.manufacturer AS api_sensors_manufacturer, api_sensors.model AS api_sensors_model, api_sensors.firmware_version AS api_sensors_

,sensor_name,type,unit,location,readings,first_reading,last_reading,is_active
6,Household power draw,Power kW,kW,Home Location,435362,2025-09-08 01:39:36.731442+00:00,2026-03-24 14:17:26.310161+00:00,True
1,idm_aero_hp_warmemenge_gesamt,Energy meter,kWh,Home Location,97154,2025-07-17 12:55:42.440820+00:00,2026-03-24 12:51:04.626018+00:00,True
4,warmepumpe_Energie_sum,Energy meter,kWh,Home Location,95402,2025-07-30 08:57:07.913676+00:00,2026-03-24 12:51:04.617639+00:00,True
3,idm_aero_hp_momentanleistung,Power kW,kW,Home Location,94551,2025-07-16 11:01:42.454120+00:00,2026-03-24 12:51:04.608111+00:00,True
5,Photovoltaics Power,Power,W,Home Location,93822,2025-09-08 04:56:37.309988+00:00,2026-03-24 14:17:25.699641+00:00,True
11,idm_aero_hp_warmemenge_heizen,Energy meter,kWh,Home Location,87187,2025-07-17 12:58:10.907176+00:00,2026-03-24 12:51:04.503141+00:00,True
2,idm_aero_hp_aktuelle_leistungsaufnahme_warmepumpe,Power kW,kW,Home Location,66363,2025-07-16 11:01:42.457199+00:00,2026-03-24 12:51:04.630734+00:00,True
12,ug_kueche_temp,Temperature,°C,Home Location,15403,2025-10-09 12:52:47.912937+00:00,2026-03-24 14:01:29.575103+00:00,True
0,wohnzimmer_temperature,Temperature,°C,Home Location,11909,2025-10-09 13:17:52.996441+00:00,2026-03-24 14:11:31.494210+00:00,True
9,idm_aero_hp_warmemenge_warmwasser,Energy meter,kWh,Home Location,8378,2025-07-17 12:58:06.832617+00:00,2026-03-24 11:48:02.744467+00:00,True


In [3]:
# Reading counts per sensor — bar chart
fig = px.bar(
    df_sensors, x='sensor_name', y='readings',
    color='type', title='Reading Count per Sensor',
    labels={'sensor_name': 'Sensor', 'readings': 'Number of Readings'},
)
fig.update_layout(xaxis_tickangle=-45, height=450)
fig.show()

## 2. Time Coverage

Visualize which sensors have data and when, to spot gaps and coverage differences.

In [4]:
# Timeline / Gantt-style view of sensor data coverage
fig = px.timeline(
    df_sensors, x_start='first_reading', x_end='last_reading',
    y='sensor_name', color='type',
    title='Sensor Data Time Coverage',
)
fig.update_yaxes(autorange='reversed')
fig.update_layout(height=500)
fig.show()

## 3. Data Density Heatmap

Readings per day per sensor — reveals gaps, outages, and sampling frequency changes.

In [5]:
# Load daily reading counts per sensor (aggregated in SQL for speed)
with get_db_session() as db:
    result = db.execute(text("""
        SELECT s.name AS sensor_name,
               DATE(sr.timestamp) AS day,
               COUNT(*) AS count
        FROM api_sensor_readings sr
        JOIN api_sensors s ON s.id = sr.sensor_id
        GROUP BY s.name, DATE(sr.timestamp)
        ORDER BY day
    """))
    df_density = pd.DataFrame(result.fetchall(), columns=['sensor_name', 'day', 'count'])

df_density['day'] = pd.to_datetime(df_density['day'])
print(f"Loaded {len(df_density)} sensor-day combinations")

# Pivot for heatmap
pivot = df_density.pivot_table(index='sensor_name', columns='day', values='count', fill_value=0)

fig = px.imshow(
    pivot, aspect='auto',
    title='Daily Reading Counts per Sensor (Heatmap)',
    labels={'x': 'Date', 'y': 'Sensor', 'color': 'Readings/day'},
    color_continuous_scale='YlOrRd',
)
fig.update_layout(height=500)
fig.show()

2026-03-24 15:17:47,785 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:17:47,787 INFO sqlalchemy.engine.Engine 
        SELECT s.name AS sensor_name,
               DATE(sr.timestamp) AS day,
               COUNT(*) AS count
        FROM api_sensor_readings sr
        JOIN api_sensors s ON s.id = sr.sensor_id
        GROUP BY s.name, DATE(sr.timestamp)
        ORDER BY day
    
2026-03-24 15:17:47,787 INFO sqlalchemy.engine.Engine [generated in 0.00056s] {}
2026-03-24 15:17:52,560 INFO sqlalchemy.engine.Engine COMMIT
Loaded 2339 sensor-day combinations


## 4. Value Distribution per Sensor Type

Box plots showing value ranges and outliers for each sensor category.

In [6]:
# Sample recent readings to show value distributions (limit per sensor for speed)
with get_db_session() as db:
    result = db.execute(text("""
        SELECT s.name AS sensor_name, st.name AS sensor_type, st.unit,
               sr.value, sr.timestamp
        FROM api_sensor_readings sr
        JOIN api_sensors s ON s.id = sr.sensor_id
        JOIN api_sensor_types st ON st.id = s.sensor_type_id
        WHERE sr.timestamp > NOW() - INTERVAL '30 days'
        ORDER BY sr.timestamp DESC
        LIMIT 50000
    """))
    df_values = pd.DataFrame(result.fetchall(),
                             columns=['sensor_name', 'sensor_type', 'unit', 'value', 'timestamp'])

print(f"Loaded {len(df_values)} recent readings")

# Box plot per sensor type
fig = px.box(
    df_values, x='sensor_name', y='value', color='sensor_type',
    title='Value Distribution per Sensor (last 30 days)',
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

2026-03-24 15:17:56,007 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:17:56,008 INFO sqlalchemy.engine.Engine 
        SELECT s.name AS sensor_name, st.name AS sensor_type, st.unit,
               sr.value, sr.timestamp
        FROM api_sensor_readings sr
        JOIN api_sensors s ON s.id = sr.sensor_id
        JOIN api_sensor_types st ON st.id = s.sensor_type_id
        WHERE sr.timestamp > NOW() - INTERVAL '30 days'
        ORDER BY sr.timestamp DESC
        LIMIT 50000
    
2026-03-24 15:17:56,010 INFO sqlalchemy.engine.Engine [generated in 0.00109s] {}
2026-03-24 15:17:56,426 INFO sqlalchemy.engine.Engine COMMIT
Loaded 50000 recent readings


## 5. Data Quality Check

Identify null values, duplicate timestamps, and potential outliers.

In [7]:
with get_db_session() as db:
    # Null values
    null_count = db.query(func.count(SensorReading.id)).filter(SensorReading.value == None).scalar()

    # Duplicate timestamps per sensor
    dupes = db.execute(text("""
        SELECT s.name, COUNT(*) AS dupe_count
        FROM (
            SELECT sensor_id, timestamp, COUNT(*) AS cnt
            FROM api_sensor_readings
            GROUP BY sensor_id, timestamp
            HAVING COUNT(*) > 1
        ) d
        JOIN api_sensors s ON s.id = d.sensor_id
        GROUP BY s.name
    """))
    df_dupes = pd.DataFrame(dupes.fetchall(), columns=['sensor_name', 'duplicate_timestamps'])

    # Total readings
    total = db.query(func.count(SensorReading.id)).scalar()

print(f"Total readings: {total:,}")
print(f"Null values: {null_count:,}")
print(f"\nDuplicate timestamps per sensor:")
if len(df_dupes):
    display(df_dupes)
else:
    print("  None found ✅")

2026-03-24 15:18:00,130 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:18:00,133 INFO sqlalchemy.engine.Engine SELECT count(api_sensor_readings.id) AS count_1 
FROM api_sensor_readings 
WHERE api_sensor_readings.value IS NULL
2026-03-24 15:18:00,133 INFO sqlalchemy.engine.Engine [generated in 0.00059s] {}
2026-03-24 15:18:00,407 INFO sqlalchemy.engine.Engine 
        SELECT s.name, COUNT(*) AS dupe_count
        FROM (
            SELECT sensor_id, timestamp, COUNT(*) AS cnt
            FROM api_sensor_readings
            GROUP BY sensor_id, timestamp
            HAVING COUNT(*) > 1
        ) d
        JOIN api_sensors s ON s.id = d.sensor_id
        GROUP BY s.name
    
2026-03-24 15:18:00,408 INFO sqlalchemy.engine.Engine [generated in 0.00090s] {}
2026-03-24 15:18:02,881 INFO sqlalchemy.engine.Engine SELECT count(api_sensor_readings.id) AS count_1 
FROM api_sensor_readings
2026-03-24 15:18:02,882 INFO sqlalchemy.engine.Engine [generated in 0.00073s] {}
2026-03-24 15:1